# Read policy signs from rendered observations

**Evidence state:** Defined

## What this demonstrates

Render a bounded arcade state as pixels, address a calibrated visual index, and recover the corresponding policy action through `VisualSignReader`.

## Why it matters

Observation, visual addressing, policy storage, and action selection remain separate and inspectable.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "VERSION").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("ZeroModel repository root not found")
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "examples"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"repository root: {ROOT}")


## Source and package mapping

- `examples/arcade_visual_sign_reader.py`
- `examples/arcade_shooter_policy.py`
- `zeromodel-vision`
- `zeromodel-video`


In [ ]:
import json
from IPython.display import Image, display
from examples.arcade_visual_sign_reader import ShooterConfig, compile_policy_artifact, compile_visual_index_artifact, make_visual_reader, render_state_frame, run_visual_policy_episode
from zeromodel.core import png_bytes

config = ShooterConfig()
policy = compile_policy_artifact(config)
visual_build = compile_visual_index_artifact(config, policy_artifact=policy)
reader = make_visual_reader(policy, visual_build)
frame = render_state_frame(0, 2, 0, width=config.width)
display(Image(data=png_bytes(frame.astype("float64") / 255.0), width=420))
decision = reader.read(frame)
print(json.dumps({"policy_artifact_id": policy.artifact_id, "visual_index_artifact_id": visual_build.artifact.artifact_id, "decision": decision.to_dict()}, indent=2, sort_keys=True))


## Application

```text
rendered observation -> feature transform -> calibrated visual index -> policy row -> action plus evidence
```


In [ ]:
episode = run_visual_policy_episode(config, policy_artifact=policy, visual_index_build=visual_build, visual_reader=reader)
print(json.dumps({"cleared": episode["cleared"], "score": episode["score"], "steps": episode["steps"], "first_decision": episode["trace"][0]["visual_decision"]}, indent=2))


## Boundaries and limitations

The arcade fixture provides bounded symbolic ground truth. This fast notebook does not run the exhaustive sweep and does not establish robustness to unseen or open-world imagery.

## Reproduction record

The builder records execution metadata and HTML under `docs/results/demos/visual-sign-reader/`.


In [ ]:
print(json.dumps({"demo_id": "visual-sign-reader", "policy_artifact_id": policy.artifact_id, "visual_index_artifact_id": visual_build.artifact.artifact_id}, indent=2))
